# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [11]:
# Re-installing to ensure dependencies are in the environment path
!pip install -U -q mcp nest_asyncio requests azure-ai-inference
!npm install -g @openbnb/mcp-server-airbnb

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧npm warn deprecated whatwg-encoding@3.1.1: Use @exodus/bytes instead for a more spec-conformant and faster implementation
⠧⠇⠏⠋⠙⠹npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead
⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 124 packages in 6s
⠦
⠦51 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [12]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout → 1, stderr → 2
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [13]:

import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_AIRBNB = True  # True if npm server available
USE_REAL_LLM = True     # True if GITHUB_TOKEN set


In [14]:
import os
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [16]:
import os
from google.colab import userdata  # Colab secrets API

# If your secret is saved under the key "GITHUB_TOKEN" in Colab:
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

# If you used a different key name in the secrets UI, e.g. "github_token":
# os.environ["GITHUB_TOKEN"] = userdata.get("github_token")

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))


GITHUB_TOKEN visible to Python: True


## Local notes MCP server

In [17]:
from pathlib import Path

# Define the filename for our local MCP server
LOCAL_SERVER = Path("local_notes_server.py")

# Write the server code to a local file
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP
notes = []

# Initialize the FastMCP server for local notes
mcp = FastMCP("LocalNotes")

# Register the add_note function as an MCP tool
@mcp.tool()
def add_note(text: str) -> str:
    """Add a note to the in-memory list."""
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

# Register the list_notes function as an MCP tool
@mcp.tool()
def list_notes() -> str:
    """List saved notes."""
    if not notes:
        return "No notes yet"
    return " ".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    mcp.run()
'''.strip(),
    encoding="utf-8",
)
print("Local server script written to:", LOCAL_SERVER)

Local server script written to: local_notes_server.py


## Client helpers (convert tools, stub planner, optional real LLM)

In [18]:
import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List

# Ensure library is loaded after install
try:
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mcp"])
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str):
    """Format MCP tool metadata for Azure LLM compatibility."""
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    """Call the LLM to process the prompt and decide on tool usage."""
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN to proceed.")

    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))

    # Execute the request with tool definitions
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        tools=functions,
        tool_choice="auto"
    )

    calls = []
    msg = resp.choices[0].message
    if msg.tool_calls:
        for tc in msg.tool_calls:
            args = tc.function.arguments
            args_json = json.loads(args) if isinstance(args, str) else args
            calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [19]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    import os
    import json

    # MINIMAL FIX: shrink tool_results before sending to gpt-4o
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )


    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        # use the shrunk version here
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestrate (connect both servers and execute tool_calls)

In [20]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def orchestrate(prompt: str):
    """Connect to MCP servers and execute tool calls requested by the LLM."""
    # Setup parameters for local notes server
    local_params = StdioServerParameters(
        command="python3",
        args=[str(LOCAL_SERVER)],
        env=BASE_ENV,
    )

    # Setup parameters for Airbnb server
    airbnb_params = StdioServerParameters(
        command="npx",
        args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
        env=BASE_ENV,
    )

    # Open connections to both servers using stdio
    async with stdio_client(local_params) as (lr, lw):
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = await local_sess.list_tools()

            async with stdio_client(airbnb_params) as (ar, aw):
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = await airbnb_sess.list_tools()

                    # Consolidate available tools from both sources
                    functions = (
                        [convert_tool(t, "notes") for t in local_tools.tools]
                        + [convert_tool(t, "airbnb") for t in airbnb_tools.tools]
                    )

                    # Ask LLM what to do
                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)

                    tool_results = []
                    for call in tool_calls:
                        name = call["name"]
                        args = call["args"]
                        prefix, tool_name = name.split("__", 1)

                        # Dispatch call to the appropriate session
                        if prefix == "notes":
                            res = await local_sess.call_tool(tool_name, args)
                        elif prefix == "airbnb":
                            res = await airbnb_sess.call_tool(tool_name, args)

                        tool_results.append({
                            "name": name,
                            "args": args,
                            "content": [c.text for c in res.content if hasattr(c, "text")]
                        })

                    return tool_calls, tool_results

## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [21]:
import asyncio

# The final demo script to run the orchestration workflow
prompt = "What tools can you access? list them please"

try:
    # Run the async orchestrator
    tool_calls, tool_results = asyncio.run(orchestrate(prompt))

    # Generate the final user-facing answer
    final_response = answer_with_llm(prompt, tool_calls, tool_results)

    print("--- Execution Success ---")
    print("Tool Calls:", tool_calls)
    print("\nFinal Response:\n", final_response)
except Exception as e:
    print(f"An error occurred during orchestration: {e}")

--- Execution Success ---
Tool Calls: []

Final Response:
 I cannot directly list the tools I have access to, as no tools were called in this interaction. However, feel free to ask about a specific type of tool or functionality, and I can let you know if I can assist with it.

## Tools used
- None
